In [ ]:
import QuantLib as ql
import plotly.express as px

In [ ]:
import numpy as np
import polars as pl

In [ ]:
layout_dict = dict(
    margin=dict(l=20, r=20, t=40, b=20),
    width=600,
    height=400,
    paper_bgcolor="LightSteelBlue",
    title_font_size=14,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
)

In [ ]:
def get_sample_bond() -> ql.CallableFixedRateBond:
    # bond creation
    settlement_days = 2
    face_amount = 100

    issue_date = ql.Date(30, 6, 2019)
    maturity_date = ql.Date(15, 9, 2023)
    tenor = ql.Period(ql.Semiannual)
    calendar = ql.UnitedStates(ql.UnitedStates.Settlement)
    accrual_convention = ql.Unadjusted
    Rule = ql.DateGeneration.Backward
    endofMonth = False
    schedule = ql.Schedule(
        issue_date,
        maturity_date,
        tenor,
        calendar,
        accrual_convention,
        accrual_convention,
        Rule,
        endofMonth,
    )

    coupon = 0.07
    day_count = ql.Actual360()  # ql.ActualActual(ql.ActualActual.Bond)

    callability_schedule = ql.CallabilitySchedule()
    for strike, dt in zip(
        [106.0, 104.0, 102.0, 100.0, 100.0],
        [
            ql.Date(15, 9, 2019),
            ql.Date(15, 9, 2020),
            ql.Date(15, 9, 2021),
            ql.Date(15, 9, 2022),
            ql.Date(15, 9, 2023),
        ],
    ):
        callability_schedule.append(
            ql.Callability(
                ql.BondPrice(strike, ql.BondPrice.Clean), ql.Callability.Call, dt
            )
        )

    bond = ql.CallableFixedRateBond(
        settlement_days,
        face_amount,
        schedule,
        [coupon],
        day_count,
        ql.Following,
        face_amount,  # redemption
        issue_date,
        callability_schedule,
    )

    return bond

In [ ]:
def calc_asof(bond, calc_date: ql.Date, oas: float | None=None) -> dict:
    # calc_date = ql.Date(30, 6, 2019)
    ql.Settings.instance().evaluationDate = calc_date

    # bond valuation
    # HullWhite dX = a*(theta - r) + s*dW, theta is term structure..
    a = 0.2
    sigma = 0.05
    grid_points = 200
    flat_rate = 0.02

    settlement_days = bond.settlementDays()
    curve = ql.FlatForward(
        settlement_days,
        ql.TARGET(),
        ql.QuoteHandle(ql.SimpleQuote(flat_rate)),
        ql.Actual360(),
    )
    ts_handle = ql.YieldTermStructureHandle(curve)

    model = ql.HullWhite(ts_handle, a, sigma)
    engine = ql.TreeCallableFixedRateBondEngine(model, grid_points)
    bond.setPricingEngine(engine)

    day_count = ql.Actual360()
    compounding = ql.Compounded
    frequency = ql.Annual

    # bond_yield = bond.bondYield(
    #     ql.BondPrice(100, ql.BondPrice.Clean), day_count, compounding, frequency
    # )
    if oas is None:
        oas = bond.OAS(100.0, ts_handle, day_count, compounding, frequency)
    px = bond.cleanPriceOAS(oas, ts_handle, day_count, compounding, frequency)

    # as of what date?
    zs = ql.BondFunctions.zSpread(bond, ql.BondPrice(px, ql.BondPrice.Clean), curve, ql.Actual360(), ql.Compounded, ql.Annual)

    return {'oas':oas, 'zspread': zs, 'price':px}

In [ ]:
bond = get_sample_bond()
calc_asof(bond, ql.Date(30, 6, 2019), 0.0070)

In [ ]:
# zspread
bond = get_sample_bond()
oas = np.linspace(-0.03 + 1e-4, 0.21, 600)
prices = np.empty(600, float)
for i, o in enumerate(oas):
    prices[i] = calc_asof(bond, ql.Date(30, 6, 2019), o)['price']
fig = px.line(pl.DataFrame(dict(oas=oas, price=prices)), "oas", "price")
fig.update_layout(**layout_dict)

In [ ]:
# issue_date = ql.Date(30, 6, 2019)
# maturity_date = ql.Date(15, 9, 2023)
bond = get_sample_bond()
oas = 0.0070
sched = ql.MakeSchedule(ql.Date(30, 6, 2019), ql.Date(15, 9, 2023), ql.Period('1D'), calendar=ql.UnitedStates(ql.UnitedStates.Settlement))

dates = np.empty(len(sched), dtype='datetime64[D]')
prices = np.empty(len(sched), float)
zspreads = np.empty(len(sched), float)

for i, d in enumerate(sched):
    dates[i] = np.datetime64(d.to_date())
    try:
        res = calc_asof(bond, d, oas)
        prices[i] =res['price']
        zspreads[i] =res['zspread']
    except RuntimeError:
        print(d)

In [ ]:
fig = px.line(pl.DataFrame(dict(date = dates, price = prices, zspread=zspreads)),x = 'date', y ='zspread')
fig.update_layout(**layout_dict)cv

In [ ]:
fig = px.line(pl.DataFrame(dict(date = dates, price = prices, zspread=zspreads)), x = 'date', y ='price')
fig.update_layout(**layout_dict)